# EmpowerLens — Cascade runner (single backbone, disk-safe, timeout-safe, resumable)

**Third revision.** History: run 1 failed on disk space (18 configs, no cleanup). Run 2
fixed disk but hung for 11.4 hours inside a single `evaluate.py` call (seed 1337 binary,
stuck right after model loading) and got killed by Kaggle's 12-hour hard session limit --
with nothing to show for it, since results only synced to `/kaggle/working/` once per
*stage*, not per seed.

This revision adds three more fixes on top of the disk-safety fixes from run 2:

1. **Pinned to a single GPU** (`CUDA_VISIBLE_DEVICES=0`). Kaggle assigned `T4 x2`; the
   leading suspect for the hang is `transformers`/`accelerate` trying to coordinate
   weight-loading across both visible devices during `from_pretrained()`, even though
   nothing here intentionally uses 2 GPUs.
2. **Every subprocess call has a hard timeout** (40 min train / 15 min eval) and gets
   killed via its process group if it hangs, instead of blocking forever. Worst case,
   one stuck config now costs under an hour, not your whole session.
3. **Results sync to `/kaggle/working/` after every single seed**, not once per stage --
   and a config already synced is detected and skipped if you re-run this notebook after
   a restart, so nothing already-completed gets retrained.

One backbone only — `mental/mental-roberta-base` — beat DeBERTa-v3-base on every metric
in the prior side-by-side run and had a much smaller val→test overfitting gap. 9 configs
total (3 tasks × 3 seeds).

**Before running:**
1. Settings -> **Accelerator: GPU**, **Internet: On**.
2. `data/splits_combined/` already committed and pushed on the branch below.
3. `src/losses.py`, `src/make_splits_cascade.py`, `src/evaluate_cascade.py`, and the
   updated `src/train_transformer.py` pushed to the branch too.
4. `HF_TOKEN` Kaggle secret set (mental-roberta-base is gated on the Hub).

**If this session also dies partway through:** just re-run the whole notebook top to
bottom on a fresh session. Every already-completed seed is detected via its eval JSON
and skipped instantly instead of retraining -- you only lose time on the config that
was actually running when it died.

In [1]:
import os

# 1. Safely reset the working directory to the Kaggle root
os.chdir('/kaggle/working/')
!rm -rf /kaggle/working/empowerlens  # <-- ONLY delete the code repo, not your checkpoints!

# 2. Clone the repo from YOUR newly merged branch
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "izza-space"

!git clone --branch $BRANCH $REPO_URL empowerlens
os.chdir('/kaggle/working/empowerlens')

# 3. Install the transformer stack AND captum
!pip install --upgrade pip setuptools wheel
!pip install -q -r requirements-transformer.txt
!pip install -q sentencepiece protobuf captum

Cloning into 'empowerlens'...
remote: Enumerating objects: 381, done.
remote: Counting objects: 100% (381/381), done.
remote: Compressing objects: 100% (279/279), done.
remote: Total 381 (delta 172), reused 303 (delta 94), pack-reused 0 (from 0)
Receiving objects: 100% (381/381), 7.94 MiB | 19.27 MiB/s, done.
Resolving deltas: 100% (172/172), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 41.6 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build depende

In [2]:
# 1a. mental/mental-roberta-base is GATED on the Hub — accept its terms at
#     https://huggingface.co/mental/mental-roberta-base while logged in, then add a
#     Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) with a read-scope HF token.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"[warn] no working HF_TOKEN secret ({e}) — training will 401 until this is set.")

Logged in to Hugging Face Hub.


In [3]:
# 1a2. Pin to a SINGLE GPU. Kaggle sometimes assigns T4 x2 -- transformers/accelerate's
#      weight-loading path can try to coordinate across all visible CUDA devices during
#      from_pretrained() even though nothing here intentionally uses 2 GPUs, and on some
#      driver/container combos that coordination deadlocks silently (this is the leading
#      suspect for the 11-hour hang seen at seed 1337's evaluate.py call last run).
# MUST run before torch/transformers are imported by any subprocess -- os.environ set here
# propagates to every `python -m src....` subprocess.run() call below since they inherit
# this process's environment.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-590dc8ce-4b88-7317-ba54-f77f17a6c0b2)
GPU 1: Tesla T4 (UUID: GPU-d034b969-a593-4f19-e908-0be791c63d5c)


In [4]:
# 1c. Shared helpers used by every stage below:
#     - sh(cmd, timeout): runs a command with a HARD timeout. If it hangs past the
#       timeout, the whole process group is killed and we move on -- no single stuck
#       command can eat the rest of the 12-hour session again like it just did.
#     - run_and_report(...): trains one config, evaluates it, deletes the wasteful
#       checkpoint-XXX/ optimizer-state subfolder, prints test metrics immediately,
#       and syncs to /kaggle/working/ after EVERY seed (not just at the end of a
#       stage) -- so a hang on seed 2 doesn't cost you seed 1's already-done work.
#       Also SKIPS a config if its eval JSON already exists, so re-running this cell
#       after a restart doesn't redundantly retrain configs you already have.
import json
import os
import signal
import subprocess
import time
from pathlib import Path

MODEL = "mental/mental-roberta-base"
TAG = MODEL.split("/")[-1]
SEEDS = (42, 1337, 2024)
TRAIN_TIMEOUT = 3600   # 1 hr 
EVAL_TIMEOUT = 3600     # 1 hr becasue it keeps timing out

def sh(cmd, timeout=None):
    print(f"$ {cmd}")
    proc = subprocess.Popen(cmd, shell=True, start_new_session=True)
    try:
        rc = proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"[TIMEOUT after {timeout}s] killing process group: {cmd}")
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except ProcessLookupError:
            pass
        proc.wait()
        rc = -1
    if rc != 0:
        print(f"[FAILED] exit code {rc}: {cmd}")
    return rc

def sync(folder):
    Path(f"/kaggle/working/{folder}").mkdir(parents=True, exist_ok=True)
    sh(f"cp -r {folder}/* /kaggle/working/{folder}/")

def show_test_metrics(out_dir, task, seed):
    p = Path(out_dir) / f"eval_{TAG}_{task}_{seed}.json"
    if not p.exists():
        print(f"  [seed {seed}] eval JSON not found at {p} — train/evaluate step failed or timed out above")
        return
    m = json.loads(p.read_text())["splits"]["test"]["metrics"]
    if task == "binary":
        print(f"  [seed {seed}] test weighted_f1={m['weighted_f1']:.3f}  positive_class_f1={m['positive_class_f1']:.3f}")
    elif task == "multiclass":
        print(f"  [seed {seed}] test weighted_f1={m['weighted_f1']:.3f}  macro_f1_10={m['macro_f1_10']:.3f}")
    else:
        print(f"  [seed {seed}] test weighted_f1={m['weighted_f1']:.3f}  macro_f1={m['macro_f1']:.3f}")

def run_and_report(task, splits_dir, out_dir, seed, extra_flags=""):
    ckpt = f"checkpoints/{task}_{TAG}_{seed}"
    eval_json = Path(out_dir) / f"eval_{TAG}_{task}_{seed}.json"

    if eval_json.exists():
        print(f"  [seed {seed}] already completed (found {eval_json}) — skipping")
        show_test_metrics(out_dir, task, seed)
        return ckpt

    # Fixed Indentation Here
    rc = sh(
        f"CUDA_VISIBLE_DEVICES=0 python -m src.train_transformer --task {task} --model {MODEL} --seed {seed} "
        f"--device auto --splits {splits_dir} {extra_flags}",
        timeout=TRAIN_TIMEOUT,
    )
    
    if rc == 0:
        print("Waiting 15 seconds for GPU VRAM to flush...")
        time.sleep(15)
        rc = sh(
            f"CUDA_VISIBLE_DEVICES=0 python -m src.evaluate --checkpoint {ckpt} --reference --splits {splits_dir} --out {out_dir}",
            timeout=EVAL_TIMEOUT,
        )
        
    # Delete the Trainer's internal checkpoint-XXX/ (model+optimizer+scheduler) regardless
    # of success/failure/timeout above -- it's never read by evaluate.py, only ckpt root is.
    sh(f"rm -rf {ckpt}/checkpoint-*")
    show_test_metrics(out_dir, task, seed)
    sync(out_dir)  # persist THIS seed's result now, don't wait for the whole stage to finish
    return ckpt

In [5]:
# Step 1 — derive Stage 2's distorted-only splits from data/splits_combined.
COMBINED_SPLITS = "data/splits_combined"
STAGE2_SPLITS   = "data/splits_stage2"

!python -m src.make_splits_cascade --source $COMBINED_SPLITS --out $STAGE2_SPLITS --force

{
  "created_utc": "2026-08-14T18:43:02.986395+00:00",
  "purpose": "Stage 2 splits = --source splits filtered to y_bin==1 (distorted only). No re-splitting: row-level train/val/test boundaries are inherited unchanged from --source.",
  "source_dir": "data/splits_combined",
  "n_per_split": {
    "train": 2694,
    "val": 158,
    "test": 161
  },
  "n_removed_no_distortion": {
    "train": 1951,
    "val": 95,
    "test": 92
  },
  "per_class_train": {
    "emotional_reasoning": 347,
    "overgeneralization": 431,
    "mental_filter": 240,
    "should_statements": 324,
    "all_or_nothing": 244,
    "mind_reading": 465,
    "fortune_telling": 390,
    "magnification": 336,
    "personalization": 254,
    "labeling": 487
  },
  "source_file_sha256": {
    "train": "ef6eedbc773ff3965654a317b786b37a32fae23225b8968028a18f4782883357",
    "val": "ca10cf0b7a12f6aa642a927b3e4b26ddca55b51eff627a0028cb72f941c21f24",
    "test": "e8b757c52be9b971d27ff95e4611c358b5c72af9425b16b8af1e7069c0c47e93"

## Step 2 — Stage 1: binary model, 3 seeds

Trained on the FULL `data/splits_combined`. Results print inline as each seed finishes; synced to `/kaggle/working/results_stage1/` right after.

In [7]:
STAGE1_OUT = "results_stage1"
!mkdir -p $STAGE1_OUT

print(f"=== Stage 1 binary: {MODEL} ===")
for seed in SEEDS:
    # Use only the flags supported by your train_transformer.py parser
    extra_args = "--batch-size 32 --epochs 3" 
    run_and_report("binary", COMBINED_SPLITS, STAGE1_OUT, seed, extra_flags=extra_args)

sync(STAGE1_OUT)
!df -h /kaggle/working

=== Stage 1 binary: mental/mental-roberta-base ===
$ CUDA_VISIBLE_DEVICES=0 python -m src.train_transformer --task binary --model mental/mental-roberta-base --seed 42 --device auto --splits data/splits_combined --batch-size 32 --epochs 3
[binary] device=cuda model=mental/mental-roberta-base train=4645 val=253 epochs=3


KeyboardInterrupt: 

## Step 2b — Multiclass (11-class), 3 seeds — comparison track, not part of the cascade

Same data as Step 2. Benefits from two fixes not present in the earlier `results_combined` run: `metric_for_best_model` now selects on `macro_f1_10` instead of `macro_f1`, and `--label-smoothing 0.1` hedges against the CODIPAS/Annotated label-definition mismatch. Written to `results_multiclass_v2/` so the old numbers aren't overwritten.

In [10]:
MULTICLASS_OUT = "results_multiclass_v2"
!mkdir -p $MULTICLASS_OUT

print(f"=== Multiclass (11-class): {MODEL} ===")
for seed in SEEDS:
    run_and_report(
        "multiclass", COMBINED_SPLITS, MULTICLASS_OUT, seed,
        extra_flags="--label-smoothing 0.1 --lr-scheduler cosine --early-stopping-patience 2",
    )

sync(MULTICLASS_OUT)
!df -h /kaggle/working

=== Multiclass (11-class): mental/mental-roberta-base ===
$ CUDA_VISIBLE_DEVICES=0 python -m src.train_transformer --task multiclass --model mental/mental-roberta-base --seed 42 --device auto --splits data/splits_combined --label-smoothing 0.1 --lr-scheduler cosine --early-stopping-patience 2
[multiclass] device=cuda model=mental/mental-roberta-base train=4645 val=253 epochs=4


KeyboardInterrupt: 

## Step 3 — Stage 2: multilabel head, distorted-only, 3 seeds

Trained on `data/splits_stage2` — never sees `no_distortion`, so `macro_f1` here is the honest "can it tell the 10 types apart" number. Uses focal loss + layer-wise LR decay, the two new levers for the minority classes (`all_or_nothing`, `mental_filter`, `personalization`).

In [4]:
STAGE2_OUT = "results_stage2"
!mkdir -p $STAGE2_OUT

print(f"=== Stage 2 multilabel: {MODEL} (focal + LLRD) ===")
for seed in SEEDS:
    run_and_report(
        "multilabel", STAGE2_SPLITS, STAGE2_OUT, seed,
        extra_flags=(
            "--loss focal --focal-gamma 2.0 --llrd --llrd-decay 0.9 --lr 3e-5 "
            "--lr-scheduler cosine --grad-accum 2 --early-stopping-patience 2"
        ),
    )

sync(STAGE2_OUT)
!df -h /kaggle/working

=== Stage 2 multilabel: mental/mental-roberta-base (focal + LLRD) ===


NameError: name 'STAGE2_SPLITS' is not defined

## Step 4 — end-to-end cascade evaluation (the number that counts)

`results_stage2/` above scores Stage 2 in isolation on distorted-only inputs — that looks better than reality since it never sees Stage 1's false negatives. This chains Stage 1 -> Stage 2 and scores the composed prediction against the FULL val/test set.

In [ ]:
CASCADE_OUT = "results_cascade"
!mkdir -p $CASCADE_OUT

print(f"=== Cascade eval: {TAG} (Stage 1 + Stage 2, matched seeds) ===")
for seed in SEEDS:
    stage1_ckpt = f"checkpoints/binary_{TAG}_{seed}"
    stage2_ckpt = f"checkpoints/multilabel_{TAG}_{seed}"
    sh(
        f"python -m src.evaluate_cascade --stage1-checkpoint {stage1_ckpt} "
        f"--stage2-checkpoint {stage2_ckpt} --splits {COMBINED_SPLITS} --out {CASCADE_OUT}"
    )

sync(CASCADE_OUT)
!df -h /kaggle/working

## Step 5 — compare: multilabel flat vs cascade, multiclass old vs fixed

In [ ]:
import pandas as pd

SOURCES = {
    "results_combined (flat, old)": "results_combined",
    "results_cascade (composed)": CASCADE_OUT,
    "results_multiclass_v2 (fixed)": MULTICLASS_OUT,
}

frames = []
for label, folder in SOURCES.items():
    p = Path(folder) / "paper_comparison.csv"
    if p.exists():
        d = pd.read_csv(p)
        d["results_dir"] = label
        frames.append(d)
    else:
        print(f"[skip] {p} not found")

all_results = pd.concat(frames, ignore_index=True)

view_ml = all_results[(all_results["task"] == "multilabel") & (all_results["split"] == "test")]
print("=== multilabel: flat vs cascade (test) ===")
print(view_ml.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1"]].agg(["mean", "std"]).round(3))

view_mc = all_results[(all_results["task"] == "multiclass") & (all_results["split"] == "test")]
print("\n=== multiclass: old vs fixed (test) ===")
print(view_mc.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1_10"]].agg(["mean", "std"]).round(3))

all_results.to_csv(f"{CASCADE_OUT}/flat_vs_cascade_vs_multiclass_comparison.csv", index=False)
sync(CASCADE_OUT)
print(f"\nWrote and synced flat_vs_cascade_vs_multiclass_comparison.csv")

In [ ]:
# Final safety net — everything above already synced incrementally after each stage,
# this just re-confirms all four folders are present in /kaggle/working/.
for folder in (STAGE1_OUT, MULTICLASS_OUT, STAGE2_OUT, CASCADE_OUT):
    sync(folder)
!ls -la /kaggle/working
!df -h /kaggle/working

In [14]:
from pathlib import Path

for folder in ["results_stage1", "results_multiclass_v2", "results_stage2", "results_cascade"]:
    p = Path(f"/kaggle/working/{folder}")
    if p.exists():
        files = list(p.glob("*.json"))
        print(f"📁 {folder}: {len(files)} result files found")
        for f in files:
            print(f"   - {f.name}")
    else:
        print(f"📁 {folder}: Folder does not exist yet")

📁 results_stage1: Folder does not exist yet
📁 results_multiclass_v2: Folder does not exist yet
📁 results_stage2: 0 result files found
📁 results_cascade: Folder does not exist yet
